In [0]:
conn_string = dbutils.secrets.get("pkustra555-scope", "pkustra555-eventhub-cs")
LOGIN = dbutils.widgets.get("login")
CATALOG = dbutils.widgets.get("catalog")
EH_NAME = "pkustra555_evh"

In [0]:
namespace_name = dbutils.widgets.get("namespace_name")
bootstrap_servers= f"{namespace_name}.servicebus.windows.net:9093"
sasl_config =("kafkashaded.org.apache.kafka.common.security.plain."
        "PlainLoginModule required "
        'username="$ConnectionString" ' 
        f'password="{conn_string}";')


In [0]:
kafka_options = {
  "kafka.bootstrap.servers": bootstrap_servers,
  "subscribe": eh_name,
  "kafka.security.protocol": "SASL_SSL",
  "kafka.sasl.mechanism": "PLAIN",
  "kafka.sasl.jaas.config": sasl_config,
  "startingOffsets": "latest"
}


In [0]:
@dp.table(name = "wikipedia_bronze")
def wikipedia_bronze():
    return (
        spark.readStream
            .format("kafka")
            .options(**kafka_options)
            .load()
            .select(
                F.col("value").cast("string").alias("event_json"),
                F.col("topic"),
                F.col("partition"),
                F.col("offset"),
                F.col("timestamp").alias("eventhub_timestamp")
            )
    )